In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import json
import time
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import torch
import torch.nn as nn
from PIL import Image

from google.colab import drive
drive.mount('/content/drive/')

PROJECT_DIR = Path("/content/drive/MyDrive/Underwater-Image-Data-set-main")
V1_DIR = PROJECT_DIR / "Dataset_V1"

FINAL_DIR = V1_DIR / "Special_Analysis"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = V1_DIR / "Dataset_V1_splits.csv"
TUNING_DIR = V1_DIR / "Classical_Tuning"
MODEL_DIR = V1_DIR / "Learning_Based_Model"

df = pd.read_csv(SPLIT_FILE)
test_df = df[df["split"] == "test"].copy()

with open(TUNING_DIR / "best_validated_method.json", "r") as f:
    best_config = json.load(f)

selected_method = best_config["selected_method"]
best_parameters = best_config["best_parameters"]

print("Selected classical method:", selected_method)
print("Best validated parameters:", best_parameters)
print("Test samples:", len(test_df))

def load_rgb(path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Could not read: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def clahe_enhance(img, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(
        clipLimit=float(clip_limit),
        tileGridSize=tuple(tile_grid)
    )
    l = clahe.apply(l)
    return cv2.cvtColor(
        cv2.merge([l, a, b]),
        cv2.COLOR_LAB2RGB
    )

def gamma_enhance(img, gamma=1.3):
    img_float = img.astype(np.float32) / 255.0
    corrected = np.power(img_float, float(gamma))
    return np.clip(corrected * 255, 0, 255).astype(np.uint8)

def gray_world(img, strength=1.0):
    img_float = img.astype(np.float32)
    means = img_float.reshape(-1, 3).mean(axis=0)
    gray = means.mean()
    scale = gray / (means + 1e-6)
    corrected = np.clip(img_float * scale, 0, 255)
    result = strength * corrected + (1 - strength) * img_float
    return np.clip(result, 0, 255).astype(np.uint8)

def apply_method(img, method, params):
    method_clean = (
        method.lower()
        .replace("-", "")
        .replace("_", "")
        .replace(" ", "")
    )

    if method_clean == "clahe":
        return clahe_enhance(
            img,
            params["clip_limit"],
            tuple(params["tile_grid"])
        )

    if method_clean == "gamma":
        return gamma_enhance(
            img,
            params["gamma"]
        )

    if method_clean in ["grayworld", "grayworldwhitebalance"]:
        return gray_world(
            img,
            params["strength"]
        )

    raise ValueError(f"Unknown method: {method}")

def edge_metrics(enhanced, target):
    enhanced_gray = cv2.cvtColor(
        enhanced,
        cv2.COLOR_RGB2GRAY
    )

    target_gray = cv2.cvtColor(
        target,
        cv2.COLOR_RGB2GRAY
    )

    enhanced_edges = cv2.Canny(
        enhanced_gray,
        100,
        200
    )

    target_edges = cv2.Canny(
        target_gray,
        100,
        200
    )

    enhanced_binary = enhanced_edges > 0
    target_binary = target_edges > 0

    intersection = np.logical_and(
        enhanced_binary,
        target_binary
    ).sum()

    union = np.logical_or(
        enhanced_binary,
        target_binary
    ).sum()

    precision = (
        intersection / enhanced_binary.sum()
        if enhanced_binary.sum() > 0 else 0
    )

    recall = (
        intersection / target_binary.sum()
        if target_binary.sum() > 0 else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    preservation = (
        intersection / target_binary.sum()
        if target_binary.sum() > 0 else 0
    )

    return preservation, precision, recall, f1

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        self.output = nn.Conv2d(
            32,
            3,
            1
        )

    def forward(self, x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        b = self.bottleneck(
            self.pool3(e3)
        )

        d3 = self.up3(b)
        d3 = torch.cat(
            [d3, e3],
            dim=1
        )
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat(
            [d2, e2],
            dim=1
        )
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat(
            [d1, e1],
            dim=1
        )
        d1 = self.dec1(d1)

        return torch.sigmoid(
            self.output(d1)
        )

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model_file = MODEL_DIR / "best_unet_model.pth"

unet = None

if model_file.exists():

    unet = UNet().to(device)

    checkpoint = torch.load(
        model_file,
        map_location=device
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        unet.load_state_dict(
            checkpoint["model_state_dict"]
        )
    else:
        unet.load_state_dict(checkpoint)

    unet.eval()

    print("U-Net loaded successfully.")

else:
    print("U-Net model file not found.")
    print("Classical-only analysis will still run.")

def prepare_unet_input(img):
    resized = cv2.resize(
        img,
        (224, 224)
    )

    tensor = (
        torch.from_numpy(
            resized.astype(np.float32) / 255.0
        )
        .permute(2, 0, 1)
        .unsqueeze(0)
    )

    return tensor.to(device)

def evaluate_pair(enhanced, target):

    psnr = peak_signal_noise_ratio(
        target,
        enhanced,
        data_range=255
    )

    ssim = structural_similarity(
        target,
        enhanced,
        channel_axis=2,
        data_range=255
    )

    preservation, precision, recall, f1 = edge_metrics(
        enhanced,
        target
    )

    return {
        "PSNR": psnr,
        "SSIM": ssim,
        "Edge Preservation": preservation,
        "Edge Precision": precision,
        "Edge Recall": recall,
        "Edge F1": f1
    }

comparison_results = []

for _, row in test_df.iterrows():

    input_img = load_rgb(
        PROJECT_DIR / row["input"]
    )

    target_img = load_rgb(
        PROJECT_DIR / row["target"]
    )

    classical_start = time.perf_counter()

    classical_output = apply_method(
        input_img,
        selected_method,
        best_parameters
    )

    classical_time = (
        time.perf_counter()
        - classical_start
    )

    classical_metrics = evaluate_pair(
        classical_output,
        target_img
    )

    comparison_results.append({
        "sample_id": row["sample_id"],
        "method": selected_method,
        "condition": "Best validated classical",
        "PSNR": classical_metrics["PSNR"],
        "SSIM": classical_metrics["SSIM"],
        "Edge Preservation": classical_metrics["Edge Preservation"],
        "Edge Precision": classical_metrics["Edge Precision"],
        "Edge Recall": classical_metrics["Edge Recall"],
        "Edge F1": classical_metrics["Edge F1"],
        "Processing Time": classical_time
    })

    if unet is not None:

        unet_input = prepare_unet_input(
            input_img
        )

        target_resized = cv2.resize(
            target_img,
            (224, 224)
        )

        with torch.no_grad():

            start = time.perf_counter()

            prediction = unet(
                unet_input
            )

            unet_time = (
                time.perf_counter()
                - start
            )

        unet_output = (
            prediction.squeeze(0)
            .permute(1, 2, 0)
            .cpu()
            .numpy()
            * 255
        ).clip(
            0,
            255
        ).astype(np.uint8)

        unet_metrics = evaluate_pair(
            unet_output,
            target_resized
        )

        comparison_results.append({
            "sample_id": row["sample_id"],
            "method": "U-Net",
            "condition": "Learning-based",
            "PSNR": unet_metrics["PSNR"],
            "SSIM": unet_metrics["SSIM"],
            "Edge Preservation": unet_metrics["Edge Preservation"],
            "Edge Precision": unet_metrics["Edge Precision"],
            "Edge Recall": unet_metrics["Edge Recall"],
            "Edge F1": unet_metrics["Edge F1"],
            "Processing Time": unet_time
        })

comparison_df = pd.DataFrame(
    comparison_results
)

comparison_df.to_csv(
    FINAL_DIR / "classical_vs_learning_test_results.csv",
    index=False
)

summary = (
    comparison_df
    .groupby(
        ["method", "condition"]
    )
    [
        [
            "PSNR",
            "SSIM",
            "Edge Preservation",
            "Edge Precision",
            "Edge Recall",
            "Edge F1",
            "Processing Time"
        ]
    ]
    .mean()
    .reset_index()
)

summary.to_csv(
    FINAL_DIR / "classical_vs_learning_summary.csv",
    index=False
)

ablation_results = []

method_clean = (
    selected_method
    .lower()
    .replace("-", "")
    .replace("_", "")
    .replace(" ", "")
)

if method_clean == "clahe":

    ablation_space = [
        {"clip_limit": 1.0, "tile_grid": [8, 8]},
        {"clip_limit": 2.0, "tile_grid": [8, 8]},
        {"clip_limit": 3.0, "tile_grid": [8, 8]},
        {"clip_limit": 4.0, "tile_grid": [8, 8]}
    ]

elif method_clean == "gamma":

    ablation_space = [
        {"gamma": 1.0},
        {"gamma": 1.2},
        {"gamma": 1.3},
        {"gamma": 1.5}
    ]

else:

    ablation_space = [
        {"strength": 0.5},
        {"strength": 0.75},
        {"strength": 1.0}
    ]

for params in ablation_space:

    psnr_values = []
    ssim_values = []
    edge_values = []

    for _, row in test_df.iterrows():

        input_img = load_rgb(
            PROJECT_DIR / row["input"]
        )

        target_img = load_rgb(
            PROJECT_DIR / row["target"]
        )

        output = apply_method(
            input_img,
            selected_method,
            params
        )

        metrics = evaluate_pair(
            output,
            target_img
        )

        psnr_values.append(
            metrics["PSNR"]
        )

        ssim_values.append(
            metrics["SSIM"]
        )

        edge_values.append(
            metrics["Edge Preservation"]
        )

    ablation_results.append({
        "method": selected_method,
        "parameters": json.dumps(params),
        "PSNR": np.mean(psnr_values),
        "SSIM": np.mean(ssim_values),
        "Edge Preservation": np.mean(edge_values)
    })

ablation_df = pd.DataFrame(
    ablation_results
)

ablation_df["PSNR_change_vs_best"] = (
    ablation_df["PSNR"]
    - best_config["validation_PSNR"]
)

ablation_df["SSIM_change_vs_best"] = (
    ablation_df["SSIM"]
    - best_config["validation_SSIM"]
)

ablation_df.to_csv(
    FINAL_DIR / "limited_parameter_ablation.csv",
    index=False
)

if len(summary) >= 2:

    classical_row = summary[
        summary["condition"]
        == "Best validated classical"
    ].iloc[0]

    learning_rows = summary[
        summary["condition"]
        == "Learning-based"
    ]

    if len(learning_rows) > 0:

        learning_row = learning_rows.iloc[0]

        effect = pd.DataFrame([{
            "Comparison":
                "U-Net minus classical",

            "PSNR_change":
                learning_row["PSNR"]
                - classical_row["PSNR"],

            "SSIM_change":
                learning_row["SSIM"]
                - classical_row["SSIM"],

            "Edge_Preservation_change":
                learning_row["Edge Preservation"]
                - classical_row["Edge Preservation"],

            "Edge_F1_change":
                learning_row["Edge F1"]
                - classical_row["Edge F1"],

            "Processing_Time_change":
                learning_row["Processing Time"]
                - classical_row["Processing Time"]
        }])

        effect.to_csv(
            FINAL_DIR / "quantified_classical_vs_learning_effect.csv",
            index=False
        )

print("\n" + "=" * 65)
print("SPECIAL ANALYSIS COMPLETE")
print("=" * 65)

print("\nCLASSICAL VS LEARNING")
print(summary)

print("\nPARAMETER ABLATION")
print(ablation_df)

print("\nSaved files:")
print(FINAL_DIR / "classical_vs_learning_test_results.csv")
print(FINAL_DIR / "classical_vs_learning_summary.csv")
print(FINAL_DIR / "limited_parameter_ablation.csv")

if (FINAL_DIR / "quantified_classical_vs_learning_effect.csv").exists():
    print(FINAL_DIR / "quantified_classical_vs_learning_effect.csv")

print("\nTest set was used only for final analysis.")
print("No parameter selection was performed using the test results.")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
Selected classical method: Gamma
Best validated parameters: {'gamma': 0.8}
Test samples: 48
U-Net loaded successfully.

SPECIAL ANALYSIS COMPLETE

CLASSICAL VS LEARNING
  method                 condition       PSNR      SSIM  Edge Preservation  \
0  Gamma  Best validated classical  13.844543  0.654768           0.007723   
1  U-Net            Learning-based  14.162797  0.630716           0.032845   

   Edge Precision  Edge Recall   Edge F1  Processing Time  
0        0.025546     0.007723  0.010610         0.005528  
1        0.014136     0.032845  0.017006         0.370242  

PARAMETER ABLATION
  method      parameters       PSNR      SSIM  Edge Preservation  \
0  Gamma  {"gamma": 1.0}  13.365501  0.654419           0.008117   
1  Gamma  {"gamma": 1.2}  12.763017  0.650189           0.008658   
2  Gamma  {"gamma": 1.3}  12.484187  0.647147           0.009